# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. Explore and sample the MedQuAD dataset
2. ***Generate ~3.5k medical multiple-choice questions (MCQs) using Phi-2***
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 2. Generate MCQs using Phi-2

In [16]:
import pandas as pd

## Load jsonl

In [24]:
sampled_df = pd.read_json('../data/medquad_sampled.jsonl', lines=True)
print('Shape:', sampled_df.shape)
sampled_df.head()

Shape: (3564, 2)


,question,answer
0,What causes Cowden syndrome ?,What causes Cowden syndrome? Most cases of Cow...
1,What causes Non-classic congenital adrenal hyp...,What causes non-classic congenital adrenal hyp...
2,What causes Nephrocalcinosis ?,What causes nephrocalcinosis? Nephrocalcinosis...
3,What causes Sjogren syndrome ?,What causes Sjogren syndrome? Sjogren syndrome...
4,What causes Urinary Tract Infection In Adults ?,Most UTIs are caused by bacteria that live in ...


In [30]:
# MCQ Generation from Q&A JSONL using Phi-2 on Colab Pro

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
from tqdm import tqdm
# from google.colab import files

# # Upload JSONL file with question + answer fields
# uploaded = files.upload()  # upload your qa_pairs.jsonl file
# file_path = list(uploaded.keys())[0]

file_path = "../data/medquad_sampled.jsonl"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Load and parse the uploaded JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load Phi-2 model and tokenizer
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

# Prompt template for MCQ generation
def build_prompt(question, answer):
    return (
        f"Convert the following medical Q&A into a multiple-choice question with one correct answer "
        f"and three plausible but incorrect options.\n\n"
        f"Question: {question}\n"
        f"Answer: {answer}\n\n"
        f"Multiple-choice question:"
    )

# Generate MCQs in batches
BATCH_SIZE = 8

results = []
for i in tqdm(range(0, len(qa_data), BATCH_SIZE)):
    batch = qa_data[i:i + BATCH_SIZE]
    prompts = [build_prompt(q["question"], q["answer"]) for q in batch]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    for prompt, completion in zip(prompts, decoded):
        results.append({"prompt": prompt, "completion": completion})

# Save to JSONL file
output_file = "../data/generated_mcqs.jsonl"
with open(output_file, "w") as f:
    for item in results:
        json.dump(item, f)
        f.write("\n")

print(f"Done! Generated MCQs saved to {output_file}")

Using device: mps


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: MPS backend out of memory (MPS allocated: 9.03 GB, other allocations: 464.00 KB, max allowed: 9.07 GB). Tried to allocate 250.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).